# ERK-KTR Full FOV Stimulation Pipeline
This notebook showcases how to use the ERK-KTR full FOV stimulation pipeline. The pipeline is designed to simulate the full field of view (FOV) stimulation of a cells with the ERK-KTR biosensor. As it is a demo experiment, the pipeline runs on the demo hardware provided by MicroManager. 

### Import required libraries

In [1]:
import os
import time
from rtm_pymmcore.data_structures import Channel, StimTreatment
import rtm_pymmcore.utils as utils
from pprint import pprint
import pandas as pd

In [20]:
import imaging_server_kit as sk

sk.Client("http://izbniesen.izb.unibe.ch:8002")

In [15]:
%load_ext autoreload
%autoreload 2

import rtm_pymmcore.utils as utils
from rtm_pymmcore.data_structures import Channel
from itertools import product


# Generate 20 dummy FOVs
fovs = utils.generate_dummy_fovs(n_fovs=20)

# Define imaging channels
channels = []
channels.append(Channel(name="mScarlet3", exposure=200))

# Define cell line conditions (10 mCherry, 10 Lifeact)
condition = ["mCherry"] * 10 + ["Lifeact"] * 10

# Define experiment parameters
n_fovs_per_batch = 5  # Number of FOVs per batch
imaging_interval = 15  # seconds between frames
phase_duration = 1800  # 30 minutes per phase in seconds
n_frames_per_phase = int(phase_duration / imaging_interval)

print(f"Total FOVs: {len(fovs)}")
print(f"FOVs per batch: {n_fovs_per_batch}")
print(f"Number of batches: {len(fovs) // n_fovs_per_batch}")
print(f"Each phase: {phase_duration/60} minutes ({n_frames_per_phase} frames)")

# Define stimulation parameter combinations to test
# You can manually define different combinations for each phase

exposures_to_test = [100, 250, 500]
cell_pcts_to_test = [0.1, 0.2, 0.3]
edge_dists_to_test = [0]
patterns_to_test = ["every_2nd"]


stim_combinations = [
    {
        "exposure": exposure,
        "cell_pct": cell_pct,
        "edge_dist": edge_dist,
        "pattern": pattern,
    }
    for exposure, cell_pct, edge_dist, pattern in product(
        exposures_to_test,
        cell_pcts_to_test,
        edge_dists_to_test,
        patterns_to_test,
    )
]


n_phases = len(stim_combinations)
print(f"\nTotal phases to test: {n_phases}")
print(f"Total experiment time: {n_phases * phase_duration / 3600:.2f} hours")

# Show which FOVs will be used in each phase
print(f"\n=== FOV Rotation Schedule ===")
for phase_idx in range(min(n_phases, 8)):  # Show first 8 phases as example
    fov_indices, batch_idx = utils.get_rotating_fov_indices(
        len(fovs), n_fovs_per_batch, phase_idx
    )
    print(f"Phase {phase_idx}: Batch {batch_idx} -> FOVs {fov_indices}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Total FOVs: 20
FOVs per batch: 5
Number of batches: 4
Each phase: 30.0 minutes (120 frames)

Total phases to test: 9
Total experiment time: 4.50 hours

=== FOV Rotation Schedule ===
Phase 0: Batch 0 -> FOVs [0, 1, 2, 3, 4]
Phase 1: Batch 1 -> FOVs [5, 6, 7, 8, 9]
Phase 2: Batch 2 -> FOVs [10, 11, 12, 13, 14]
Phase 3: Batch 3 -> FOVs [15, 16, 17, 18, 19]
Phase 4: Batch 0 -> FOVs [0, 1, 2, 3, 4]
Phase 5: Batch 1 -> FOVs [5, 6, 7, 8, 9]
Phase 6: Batch 2 -> FOVs [10, 11, 12, 13, 14]
Phase 7: Batch 3 -> FOVs [15, 16, 17, 18, 19]


In [16]:
# Generate df_acquire for each phase
# Each phase starts with timestep=0 and time=0
all_phases_df = []

for phase_idx in range(n_phases):  # Generate all phases
    # Get rotating FOV batch for this phase
    fov_indices, batch_idx = utils.get_rotating_fov_indices(
        len(fovs), n_fovs_per_batch, phase_idx
    )

    # Get stimulation parameters for this phase
    stim_params = stim_combinations[phase_idx]
    phase_name = f"phase{phase_idx}_batch{batch_idx}"

    print(f"\n=== Phase {phase_idx} ===")
    print(f"FOV batch {batch_idx}: FOVs {fov_indices}")
    print(f"Stim params: {stim_params}")

    # Generate df_acquire for this phase
    # Note: timestep and time always start at 0 for each phase
    df_phase = utils.generate_df_acquire_single_phase(
        fovs=fovs,
        channels=channels,
        phase_id=phase_idx,
        phase_name=phase_name,
        fov_indices=fov_indices,
        n_frames=n_frames_per_phase,
        imaging_interval=imaging_interval,
        condition=condition,
        stim_exposure=stim_params["exposure"],
        stim_cell_percentage=stim_params["cell_pct"],
        stim_edge_distance=stim_params["edge_dist"],
        stim_pattern=stim_params["pattern"],
        stim_power=10,
        stim_channel_name="CyanStim",
        stim_channel_group="TTL_ERK",
        stim_channel_device_name="Spectra",
        stim_channel_power_property_name="Cyan_Level",
    )

    all_phases_df.append(df_phase)

    print(
        f"Phase {phase_idx}: {len(df_phase)} frames, {df_phase['stim'].sum()} stim frames"
    )
    print(
        f"  Timestep range: {df_phase['timestep'].min()}-{df_phase['timestep'].max()}"
    )
    print(f"  Time range: {df_phase['time'].min()}s-{df_phase['time'].max()}s")

print(f"\n=== Summary ===")
print(f"Total phases generated: {len(all_phases_df)}")
print(f"Each phase is independent with timestep starting at 0")


=== Phase 0 ===
FOV batch 0: FOVs [0, 1, 2, 3, 4]
Stim params: {'exposure': 100, 'cell_pct': 0.1, 'edge_dist': 0, 'pattern': 'every_2nd'}
Phase 0: 600 frames, 300 stim frames
  Timestep range: 0-599
  Time range: 0s-1785s

=== Phase 1 ===
FOV batch 1: FOVs [5, 6, 7, 8, 9]
Stim params: {'exposure': 100, 'cell_pct': 0.2, 'edge_dist': 0, 'pattern': 'every_2nd'}
Phase 1: 600 frames, 300 stim frames
  Timestep range: 0-599
  Time range: 0s-1785s

=== Phase 2 ===
FOV batch 2: FOVs [10, 11, 12, 13, 14]
Stim params: {'exposure': 100, 'cell_pct': 0.3, 'edge_dist': 0, 'pattern': 'every_2nd'}
Phase 2: 600 frames, 300 stim frames
  Timestep range: 0-599
  Time range: 0s-1785s

=== Phase 3 ===
FOV batch 3: FOVs [15, 16, 17, 18, 19]
Stim params: {'exposure': 250, 'cell_pct': 0.1, 'edge_dist': 0, 'pattern': 'every_2nd'}
Phase 3: 600 frames, 300 stim frames
  Timestep range: 0-599
  Time range: 0s-1785s

=== Phase 4 ===
FOV batch 0: FOVs [0, 1, 2, 3, 4]
Stim params: {'exposure': 250, 'cell_pct': 0.2,

In [17]:
all_phases_df[2].head(10)

,fov_object,fov,fov_x,fov_y,fov_z,fov_name,timestep,fov_timestep,time,channels,...,stim_channel_name,stim_channel_group,stim_channel_device_name,stim_channel_power_property_name,stim_cell_percentage,stim_edge_distance,stim_pattern,stim,stim_timestep,stim_exposure_list
0,<rtm_pymmcore.data_structures.Fov object at 0x...,10,0,1000,None,FOV_10,0,0,0,"({'name': 'mScarlet3', 'exposure': 200, 'group...",...,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.3,0,every_2nd,True,"(0, 1, 2, 3, 4, 10, 11, 12, 13, 14, 20, 21, 22...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ..."
1,<rtm_pymmcore.data_structures.Fov object at 0x...,11,500,1000,None,FOV_11,1,0,0,"({'name': 'mScarlet3', 'exposure': 200, 'group...",...,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.3,0,every_2nd,True,"(0, 1, 2, 3, 4, 10, 11, 12, 13, 14, 20, 21, 22...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ..."
2,<rtm_pymmcore.data_structures.Fov object at 0x...,12,1000,1000,None,FOV_12,2,0,0,"({'name': 'mScarlet3', 'exposure': 200, 'group...",...,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.3,0,every_2nd,True,"(0, 1, 2, 3, 4, 10, 11, 12, 13, 14, 20, 21, 22...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ..."
3,<rtm_pymmcore.data_structures.Fov object at 0x...,13,1500,1000,None,FOV_13,3,0,0,"({'name': 'mScarlet3', 'exposure': 200, 'group...",...,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.3,0,every_2nd,True,"(0, 1, 2, 3, 4, 10, 11, 12, 13, 14, 20, 21, 22...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ..."
4,<rtm_pymmcore.data_structures.Fov object at 0x...,14,2000,1000,None,FOV_14,4,0,0,"({'name': 'mScarlet3', 'exposure': 200, 'group...",...,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.3,0,every_2nd,True,"(0, 1, 2, 3, 4, 10, 11, 12, 13, 14, 20, 21, 22...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ..."
5,<rtm_pymmcore.data_structures.Fov object at 0x...,10,0,1000,None,FOV_10,5,1,15,"({'name': 'mScarlet3', 'exposure': 200, 'group...",...,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.3,0,every_2nd,False,"(0, 1, 2, 3, 4, 10, 11, 12, 13, 14, 20, 21, 22...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ..."
6,<rtm_pymmcore.data_structures.Fov object at 0x...,11,500,1000,None,FOV_11,6,1,15,"({'name': 'mScarlet3', 'exposure': 200, 'group...",...,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.3,0,every_2nd,False,"(0, 1, 2, 3, 4, 10, 11, 12, 13, 14, 20, 21, 22...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ..."
7,<rtm_pymmcore.data_structures.Fov object at 0x...,12,1000,1000,None,FOV_12,7,1,15,"({'name': 'mScarlet3', 'exposure': 200, 'group...",...,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.3,0,every_2nd,False,"(0, 1, 2, 3, 4, 10, 11, 12, 13, 14, 20, 21, 22...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ..."
8,<rtm_pymmcore.data_structures.Fov object at 0x...,13,1500,1000,None,FOV_13,8,1,15,"({'name': 'mScarlet3', 'exposure': 200, 'group...",...,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.3,0,every_2nd,False,"(0, 1, 2, 3, 4, 10, 11, 12, 13, 14, 20, 21, 22...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ..."
9,<rtm_pymmcore.data_structures.Fov object at 0x...,14,2000,1000,None,FOV_14,9,1,15,"({'name': 'mScarlet3', 'exposure': 200, 'group...",...,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.3,0,every_2nd,False,"(0, 1, 2, 3, 4, 10, 11, 12, 13, 14, 20, 21, 22...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ..."


### Phase Summary

Display a summary table showing the stimulation parameters for each phase.

In [28]:
# Create summary DataFrame showing parameters for each phase
summary_data = []
for phase_idx in range(len(all_phases_df)):  # Only iterate over generated phases
    params = stim_combinations[phase_idx]
    fov_indices, batch_idx = utils.get_rotating_fov_indices(
        len(fovs), n_fovs_per_batch, phase_idx
    )
    df_phase = all_phases_df[phase_idx]
    summary_data.append(
        {
            "phase_id": phase_idx,
            "batch": batch_idx,
            "fov_indices": str(fov_indices),
            "n_frames": len(df_phase),
            "n_stim_frames": int(df_phase["stim"].sum()),
            "stim_exposure": params["exposure"],
            "cell_percentage": params["cell_pct"],
            "edge_distance": params["edge_dist"],
            "stim_pattern": params["pattern"],
        }
    )

df_summary = pd.DataFrame(summary_data)
print("Phase Summary:")
display(df_summary)

Phase Summary:


""


### Run Experiment Phase by Phase

Execute each phase sequentially. Each phase images a specific batch of FOVs with specific stimulation parameters.

In [3]:
import tifffile

raw = tifffile.imread("test_exp_data\\02_demo_imgs\\raw\\000_00000.tiff")

In [4]:
from rtm_pymmcore.segmentation.imaging_server import SegmentatorImagingServerKit


seg = SegmentatorImagingServerKit(
    "http://izbniesen.izb.unibe.ch:8002",
    "CellPose",
    min_size=50,
)


mask = seg.segment(raw[0])

KeyboardInterrupt: 

In [ ]:
raw.

array([[[167, 180, 163, ..., 205, 207, 257],
        [154, 197, 190, ..., 239, 204, 248],
        [146, 170, 174, ..., 260, 273, 210],
        ...,
        [214, 191, 183, ..., 192, 184, 247],
        [195, 208, 190, ..., 203, 214, 215],
        [176, 170, 182, ..., 234, 214, 193]]],
      shape=(1, 1900, 1900), dtype=uint16)

In [6]:
os.makedirs("test_exp_data\\02_demo_imgs\\segmented", exist_ok=True)
tifffile.imwrite(
    "test_exp_data\\02_demo_imgs\\segmented\\000_00000_seg.tiff", mask.astype("uint16")
)

In [6]:
mask = tifffile.imread("test_exp_data\\02_demo_imgs\\segmented\\000_00000_seg.tiff")

In [7]:
from rtm_pymmcore.stimulation.percentage_distance_edge import (
    PercentageDistanceEdgeStimulation,
)

stim = PercentageDistanceEdgeStimulation()

In [ ]:
stim.get_stim_mask(
    label_images={"labels": mask},
    metadata={"timestep": 0},
    img=raw[0],
    df_tracked=pd.DataFrame(),
)

### Experimental Settings

In [ ]:
## Configuration options - set experiment timing, storage and stimulation parameters
# General timing and frame counts:
N_FRAMES = 30 * 4  # number of timesteps
# If you want the notebook/script to wait before starting the experiment, set this (hours).
# Useful for scheduling: set to a fractional value (e.g. 0.5 for 30 minutes).
SLEEP_BEFORE_EXPERIMENT_START_in_H = 0

# Timing for acquisition: interval between timesteps (seconds) and approximate time per FOV (seconds).
# TIME_BETWEEN_TIMESTEPS: time between frames (across all FOVs).
TIME_BETWEEN_TIMESTEPS = 15  # seconds between timesteps
# TIME_PER_FOV: approximate time required to image one FOV (used for scheduling/estimates).
TIME_PER_FOV = 3.75  # seconds per FOV (camera + stage moves + overhead)

# Display / bookkeeping options:
# If True, add a column/group that stores the last stimulation exposure applied to each FOV (helpful for QC).
ADD_STIM_EXPOSURE_GROUP = False  # set to True to save per-FOV last-stim-exposure info
# When True, stim timepoints will be distributed evenly across the available timesteps rather than using explicit lists.
REGULAR_SPACING_BETWEEN_STIMULATIONS = False  # True -> evenly spaced stim timings; False -> use explicit stim_timestep lists

## Storage path for the experiment - change to your desired directory and experiment name.
base_path = "E:\\Alex"  # example: 'E:\Alex' (double-escaped for JSON/Notebook)
experiment_name = "testmigration2"
path = os.path.join(base_path, experiment_name)

## Channels: images to acquire each timestep.
# Each Channel(...) maps to a microscope channel name configured in Micro-Manager/your device.
# If exposure or power is omitted, the hardware default (set in the device/GUI) will be used.
# Examples: specify different exposures per channel, or add channels without exposures to use defaults.
channels = []
channels.append(Channel(name="mScarlet3", exposure=200))  # example: ERK-KTR reporter

# Optional channel to run an optocheck (check expression of optogenetic marker).
# If the optogenetic marker is imaged in the same channel as stimulation, you can reuse that channel here.
channel_optocheck = Channel(
    name="mCitrine", exposure=200
)  # high exposure for low-signal marker
optocheck_timepoints = (
    N_FRAMES - 1,
)  # tuple of timesteps at which to capture the optocheck channel

# Experimental condition(s): a list of labels assigned to FOVs.
# You can repeat or expand this list to match the number of FOVs.
condition = [
    "optoTIAM_single",
]
# Example alternatives (uncomment to use):
# - Repeat each condition multiple times: condition = [cond for cond in condition for _ in range(repeats)]
# - Create a long condition vector: condition = ["optoFGFR_high"] * 24 + ["optoFGFR"] * 24

# If using wellplates, set how many FOVs per well. Set to None if not using wellplates.
n_fovs_per_well = None  ## number of FOVs per well; use None for free-FOV experiments

# Stimulation parameters for optogenetics. Define a list of StimTreatment objects per phase.
# Notes on StimTreatment fields:
# - treatment_name: human-readable label for the treatment
# - stim_timestep: a tuple/list of integers (timesteps) when stimulation should occur, OR a range/tuple.
#       Examples: (10,20,30), list(range(10,100)), (tuple(range(10,100,1)))
# - stim_exposure_list: either a single exposure value (applied to all listed timesteps) or a list/tuple of exposures matching the timesteps.
# - auto_repeat_stim_exposure: when True and a single exposure value is provided, it will be repeated across timesteps.
# - stim_power/stim_channel_name/stim_channel_group: map the stimulation settings to your device/channel group.
# - stim_channel_device_name/stim_channel_power_property_name: lower-level device settings used by the hardware API.

stim_phase = [
    StimTreatment(
        treatment_name="15min_stim",
        stim_timestep=(tuple(range(1, 30 * 4, 1))),  # example: stim at timesteps 10..99
        stim_exposure_list=200,
        stim_power=10,
        stim_channel_name="CyanStim",
        stim_channel_group="TTL_ERK",
        stim_channel_device_name="Spectra",
        stim_channel_power_property_name="Cyan_Level",
        auto_repeat_stim_exposure=True,
    )
]

# a list of StimTreatment objects; when multiple are supplied they will be assigned across FOVs according to the utils.apply_stim_treatments_to_df_acquire logic

# # Print the final stim schedule for confirmation before running. Helpful to verify exposures/timings.
# for stim_phase in [
#     stim_phase,
# ]:
#     utils.print_stim_exposures_timesteps(stim_phase)

## Define the Tools that you are using for the experiment (segmentors, trackers, feature extractors, stimulator).
from rtm_pymmcore.stimulation.percentage_of_cell import StimPercentageOfCell
from rtm_pymmcore.stimulation.base_stimulation import StimWholeFOV
from rtm_pymmcore.tracking.trackpy import TrackerTrackpy
from rtm_pymmcore.feature_extraction.simple_fe import SimpleFE
from rtm_pymmcore.feature_extraction.optocheck_fe import OptoCheckFE
from rtm_pymmcore.segmentation.imaging_server import SegmentatorImagingServerKit
from rtm_pymmcore.segmentation.imaging_server import SegmentatorImagingServerKit


segmentators = [
    {
        "name": "labels",
        "class": SegmentatorImagingServerKit(
            "http://130.92.82.27:8000",
            "cellpose",
            min_size=50,
        ),
        "use_channel": 0,
        "save_tracked": True,
    },
]
stimulator = StimPercentageOfCell()
feature_extractor = SimpleFE("labels")
tracker = TrackerTrackpy()
optocheck = OptoCheckFE(used_mask="labels")


from rtm_pymmcore.img_processing_pip import ImageProcessingPipeline

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    stimulator=stimulator,
    feature_extractor_optocheck=optocheck,
)
mic.set_pipeline(pipeline=pipeline)

In [ ]:
# from rtm_pymmcore.microscope.MMDemo import MMDemo

# mic = MMDemo()
# mic.mmc.setChannelGroup("Channel")

from rtm_pymmcore.microscope.Jungfrau import Jungfrau

mic = Jungfrau()
mic.mmc.setChannelGroup("TTL_ERK")

In [ ]:
## Configuration options - set experiment timing, storage and stimulation parameters
# General timing and frame counts:
# FIRST_FRAME_STIMULATION: first frame index (0-based) when stimulation logic can start.
FIRST_FRAME_STIMULATION = 10
# N_FRAMES: (optional) total number of frames for a single long experiment. We split into phases below instead.
# N_FRAMES = 340
# Instead of a single N_FRAMES, this notebook uses phase-specific frame counts:
N_FRAMES_PHASE_1 = 100  # number of timesteps in phase 1 (pre-drug)
N_FRAMES_PHASE_2 = 150  # number of timesteps in phase 2 (post-drug)

# If you want the notebook/script to wait before starting the experiment, set this (hours).
# Useful for scheduling: set to a fractional value (e.g. 0.5 for 30 minutes).
SLEEP_BEFORE_EXPERIMENT_START_in_H = 0

# Timing for acquisition: interval between timesteps (seconds) and approximate time per FOV (seconds).
# TIME_BETWEEN_TIMESTEPS: time between frames (across all FOVs).
TIME_BETWEEN_TIMESTEPS = 60  # seconds between timesteps
# TIME_PER_FOV: approximate time required to image one FOV (used for scheduling/estimates).
TIME_PER_FOV = 3.75  # seconds per FOV (camera + stage moves + overhead)

# Display / bookkeeping options:
# If True, add a column/group that stores the last stimulation exposure applied to each FOV (helpful for QC).
ADD_STIM_EXPOSURE_GROUP = False  # set to True to save per-FOV last-stim-exposure info
# When True, stim timepoints will be distributed evenly across the available timesteps rather than using explicit lists.
REGULAR_SPACING_BETWEEN_STIMULATIONS = False  # True -> evenly spaced stim timings; False -> use explicit stim_timestep lists

## Storage path for the experiment - change to your desired directory and experiment name.
base_path = "E:\\Alex"  # example: 'E:\Alex' (double-escaped for JSON/Notebook)
experiment_name = "2025-11-13_Priming_FGFR_U0126"
path = os.path.join(base_path, experiment_name)

## Channels: images to acquire each timestep.
# Each Channel(...) maps to a microscope channel name configured in Micro-Manager/your device.
# If exposure or power is omitted, the hardware default (set in the device/GUI) will be used.
# Examples: specify different exposures per channel, or add channels without exposures to use defaults.
channels = []
channels.append(
    Channel(name="miRFP", exposure=300)
)  # example: long exposure for nuclear marker
channels.append(Channel(name="mScarlet3", exposure=200))  # example: ERK-KTR reporter

# Optional channel to run an optocheck (check expression of optogenetic marker).
# If the optogenetic marker is imaged in the same channel as stimulation, you can reuse that channel here.
channel_optocheck = Channel(
    name="mCitrine", exposure=600
)  # high exposure for low-signal marker
optocheck_timepoints = (
    149,
)  # tuple of timesteps at which to capture the optocheck channel

# Experimental condition(s): a list of labels assigned to FOVs.
# You can repeat or expand this list to match the number of FOVs.
condition = [
    "Drug",
]
# Example alternatives (uncomment to use):
# - Repeat each condition multiple times: condition = [cond for cond in condition for _ in range(repeats)]
# - Create a long condition vector: condition = ["optoFGFR_high"] * 24 + ["optoFGFR"] * 24

# If using wellplates, set how many FOVs per well. Set to None if not using wellplates.
n_fovs_per_well = None  ## number of FOVs per well; use None for free-FOV experiments

# Stimulation parameters for optogenetics. Define a list of StimTreatment objects per phase.
# Notes on StimTreatment fields:
# - treatment_name: human-readable label for the treatment
# - stim_timestep: a tuple/list of integers (timesteps) when stimulation should occur, OR a range/tuple.
#       Examples: (10,20,30), list(range(10,100)), (tuple(range(10,100,1)))
# - stim_exposure_list: either a single exposure value (applied to all listed timesteps) or a list/tuple of exposures matching the timesteps.
# - auto_repeat_stim_exposure: when True and a single exposure value is provided, it will be repeated across timesteps.
# - stim_power/stim_channel_name/stim_channel_group: map the stimulation settings to your device/channel group.
# - stim_channel_device_name/stim_channel_power_property_name: lower-level device settings used by the hardware API.

stim_phase_1 = [
    StimTreatment(
        treatment_name="Priming Phase 1 pre Drug",
        stim_timestep=(tuple(range(10, 100, 1))),  # example: stim at timesteps 10..99
        stim_exposure_list=(100,)
        * 90,  # per-timestep exposure (length must match stim_timestep)
        stim_power=10,
        stim_channel_name="CyanStim",
        stim_channel_group="TTL_ERK",
        stim_channel_device_name="Spectra",
        stim_channel_power_property_name="Cyan_Level",
    )
]

stim_phase_2 = [
    StimTreatment(
        treatment_name="Sustained Phase 2 post Drug",
        stim_timestep=(
            tuple(range(30, 120, 1))
        ),  # many ways to define timesteps: tuple, list, range, or single integer
        stim_exposure_list=100,  # single value will be repeated if auto_repeat_stim_exposure=True
        auto_repeat_stim_exposure=True,
        stim_power=10,
        stim_channel_name="CyanStim",
        stim_channel_group="TTL_ERK",
        stim_channel_device_name="Spectra",
        stim_channel_power_property_name="Cyan_Level",
    )
]  # a list of StimTreatment objects; when multiple are supplied they will be assigned across FOVs according to the utils.apply_stim_treatments_to_df_acquire logic

# Print the final stim schedule for confirmation before running. Helpful to verify exposures/timings.
for stim_phase in [
    stim_phase_1,
    stim_phase_2,
]:
    utils.print_stim_exposures_timesteps(stim_phase)

## Define the Tools that you are using for the experiment (segmentors, trackers, feature extractors, stimulator).
from rtm_pymmcore.stimulation.base_stimulation import StimWholeFOV
from rtm_pymmcore.tracking.trackpy import TrackerTrackpy
from rtm_pymmcore.feature_extraction.erk_ktr import FE_ErkKtr
from rtm_pymmcore.feature_extraction.optocheck_fe import OptoCheckFE
from rtm_pymmcore.segmentation.cellpose_v4 import CellposeV4

segmentators = [
    {
        "name": "labels",
        "class": CellposeV4(
            custom_model_path="E:\models\cellpose\LifeActH2B_mixed_with_only_H2B_v1",
            min_size=100,
        ),
        "use_channel": 0,
        "save_tracked": True,
    },
]

stimulator = StimWholeFOV()
feature_extractor = FE_ErkKtr("labels")
tracker = TrackerTrackpy(search_range=50)
optocheck = OptoCheckFE(used_mask="labels")


from rtm_pymmcore.img_processing_pip import ImageProcessingPipeline

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    stimulator=stimulator,
    feature_extractor_optocheck=optocheck,
)
mic.set_pipeline(pipeline=pipeline)

Pattern Name:  Priming Phase 1 pre Drug
100 at 10
100 at 11
100 at 12
100 at 13
100 at 14
100 at 15
100 at 16
100 at 17
100 at 18
100 at 19
100 at 20
100 at 21
100 at 22
100 at 23
100 at 24
100 at 25
100 at 26
100 at 27
100 at 28
100 at 29
100 at 30
100 at 31
100 at 32
100 at 33
100 at 34
100 at 35
100 at 36
100 at 37
100 at 38
100 at 39
100 at 40
100 at 41
100 at 42
100 at 43
100 at 44
100 at 45
100 at 46
100 at 47
100 at 48
100 at 49
100 at 50
100 at 51
100 at 52
100 at 53
100 at 54
100 at 55
100 at 56
100 at 57
100 at 58
100 at 59
100 at 60
100 at 61
100 at 62
100 at 63
100 at 64
100 at 65
100 at 66
100 at 67
100 at 68
100 at 69
100 at 70
100 at 71
100 at 72
100 at 73
100 at 74
100 at 75
100 at 76
100 at 77
100 at 78
100 at 79
100 at 80
100 at 81
100 at 82
100 at 83
100 at 84
100 at 85
100 at 86
100 at 87
100 at 88
100 at 89
100 at 90
100 at 91
100 at 92
100 at 93
100 at 94
100 at 95
100 at 96
100 at 97
100 at 98
100 at 99

Pattern Name:  Sustained Phase 2 post Drug
100 at 30
100 at

### GUI - Napari Micromanager

#### Load GUI

In [4]:
### Base GUI ###
from napari_micromanager import MainWindow
import napari

viewer = napari.Viewer()
mm_wdg = MainWindow(viewer)
mm_wdg._mmc = mic.mmc
viewer.window.add_dock_widget(mm_wdg)
data_mda_fovs = None
load_from_file = False

Please execute the following code block, if you would like to set a custom ROI (e.g. smaller illumination than the full field of view of the camera). Execute it after you have started the camera once in the GUI. 

In [ ]:
if mic.SET_ROI_REQUIRED:
    mic.mmc.clearROI()
    mic.mmc.setROI(mic.ROI_X, mic.ROI_Y, mic.ROI_WIDTH, mic.ROI_HEIGHT)

### Map Experiment to FOVs

### Use FOVs to generate dataframe for acquisition

In [6]:
fovs = utils.generate_fov_objects(mic, viewer=viewer)


df_acquire = utils.generate_df_acquire(
    fovs,
    n_frames=N_FRAMES_PHASE_1,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    time_per_fov=TIME_PER_FOV,
    channels=channels,
    phase_name="PreDrug",
    phase_id=0,
    condition=condition,
)
df_acquire = utils.apply_stim_treatments_to_df_acquire(
    df_acquire,
    stim_phase_1,
    condition,
    n_fovs_per_well=n_fovs_per_well,
    add_stim_exposure_group=ADD_STIM_EXPOSURE_GROUP,
    regular_spacing_between_stimulations=REGULAR_SPACING_BETWEEN_STIMULATIONS,
)
df_acquire

Total Experiment Time: 1.65h
Doing 3 experiment per stim condition


c:\Users\Jungfrau\Documents\alandolt\code\rtm-pymmcore\rtm_pymmcore\utils.py:177: FutureWarning: The `_dock_widgets` property is private and should not be used in any plugin code. Please use the `dock_widgets` property instead.
  data_mda_fovs = viewer.window._dock_widgets["MDA"].widget().value().stage_positions


,fov_object,fov,fov_x,fov_y,fov_name,timestep,time,channels,fname,cell_line,...,treatment_name,stim_timestep,stim_exposure_list,stim_power,stim_channel_name,stim_channel_group,stim_channel_device_name,stim_channel_power_property_name,stim_exposure,stim
0,<rtm_pymmcore.data_structures.Fov object at 0x...,0,-1.0,-0.7,0,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_00_00000,Drug,...,Priming Phase 1 pre Drug,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
1,<rtm_pymmcore.data_structures.Fov object at 0x...,1,-1.0,-0.7,1,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_00_00000,Drug,...,Priming Phase 1 pre Drug,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
2,<rtm_pymmcore.data_structures.Fov object at 0x...,2,-1.0,-0.7,2,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",002_00_00000,Drug,...,Priming Phase 1 pre Drug,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
3,<rtm_pymmcore.data_structures.Fov object at 0x...,0,-1.0,-0.7,0,1,60.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_00_00001,Drug,...,Priming Phase 1 pre Drug,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
4,<rtm_pymmcore.data_structures.Fov object at 0x...,1,-1.0,-0.7,1,1,60.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_00_00001,Drug,...,Priming Phase 1 pre Drug,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,<rtm_pymmcore.data_structures.Fov object at 0x...,1,-1.0,-0.7,1,98,5880.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_00_00098,Drug,...,Priming Phase 1 pre Drug,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,100.0,True
296,<rtm_pymmcore.data_structures.Fov object at 0x...,2,-1.0,-0.7,2,98,5880.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",002_00_00098,Drug,...,Priming Phase 1 pre Drug,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,100.0,True
297,<rtm_pymmcore.data_structures.Fov object at 0x...,0,-1.0,-0.7,0,99,5940.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_00_00099,Drug,...,Priming Phase 1 pre Drug,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,100.0,True
298,<rtm_pymmcore.data_structures.Fov object at 0x...,1,-1.0,-0.7,1,99,5940.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_00_00099,Drug,...,Priming Phase 1 pre Drug,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,100.0,True


In [7]:
df_acquire_2 = utils.generate_df_acquire(
    fovs,
    n_frames=N_FRAMES_PHASE_2,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    time_per_fov=TIME_PER_FOV,
    channels=channels,
    channel_optocheck=channel_optocheck,
    optocheck_timepoints=optocheck_timepoints,
    phase_name="PostDrug",
    phase_id=1,
    condition=condition,
)
df_acquire_2 = utils.apply_stim_treatments_to_df_acquire(
    df_acquire_2,
    stim_phase_2,
    condition,
    n_fovs_per_well=n_fovs_per_well,
    add_stim_exposure_group=ADD_STIM_EXPOSURE_GROUP,
    regular_spacing_between_stimulations=REGULAR_SPACING_BETWEEN_STIMULATIONS,
)
df_acquire_2

Total Experiment Time: 2.4833333333333334h
Doing 3 experiment per stim condition


,fov_object,fov,fov_x,fov_y,fov_name,timestep,time,channels,fname,cell_line,...,treatment_name,stim_timestep,stim_exposure_list,stim_power,stim_channel_name,stim_channel_group,stim_channel_device_name,stim_channel_power_property_name,stim_exposure,stim
0,<rtm_pymmcore.data_structures.Fov object at 0x...,0,-1.0,-0.7,0,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_01_00000,Drug,...,Sustained Phase 2 post Drug,"(30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 4...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
1,<rtm_pymmcore.data_structures.Fov object at 0x...,1,-1.0,-0.7,1,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_01_00000,Drug,...,Sustained Phase 2 post Drug,"(30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 4...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
2,<rtm_pymmcore.data_structures.Fov object at 0x...,2,-1.0,-0.7,2,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",002_01_00000,Drug,...,Sustained Phase 2 post Drug,"(30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 4...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
3,<rtm_pymmcore.data_structures.Fov object at 0x...,0,-1.0,-0.7,0,1,60.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_01_00001,Drug,...,Sustained Phase 2 post Drug,"(30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 4...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
4,<rtm_pymmcore.data_structures.Fov object at 0x...,1,-1.0,-0.7,1,1,60.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_01_00001,Drug,...,Sustained Phase 2 post Drug,"(30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 4...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
445,<rtm_pymmcore.data_structures.Fov object at 0x...,1,-1.0,-0.7,1,148,8880.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_01_00148,Drug,...,Sustained Phase 2 post Drug,"(30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 4...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
446,<rtm_pymmcore.data_structures.Fov object at 0x...,2,-1.0,-0.7,2,148,8880.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",002_01_00148,Drug,...,Sustained Phase 2 post Drug,"(30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 4...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
447,<rtm_pymmcore.data_structures.Fov object at 0x...,0,-1.0,-0.7,0,149,8940.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_01_00149,Drug,...,Sustained Phase 2 post Drug,"(30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 4...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
448,<rtm_pymmcore.data_structures.Fov object at 0x...,1,-1.0,-0.7,1,149,8940.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_01_00149,Drug,...,Sustained Phase 2 post Drug,"(30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 4...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False


### Run experiment

In [8]:
for _ in range(0, SLEEP_BEFORE_EXPERIMENT_START_in_H * 3600):
    time.sleep(1)

try:
    mm_wdg._core_link.cleanup()

except:
    pass

mic.run_experiment(df_acquire)

In [9]:
mic.run_experiment(df_acquire_2)

In [10]:
mic.post_experiment()
time.sleep(10)

utils.generate_exp_data_from_tracks(path)

from napari_micromanager._core_link import CoreViewerLink

if "viewer" in locals():
    mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

### Function to re-connect link with GUI if manually broken

The following functions can be used to manually re-make the connection between the GUI and the running rtm-pymmcore script. However, normally you don't need to execute them. 

In [18]:
### Manually reconnect pymmcore with napari-micromanager
from napari_micromanager._core_link import CoreViewerLink

mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

The following code block can be used to manually break the connection between GUI and Jupyter Notebook:


In [ ]:
### Break connection
# mm_wdg._core_link.cleanup()